In [1]:
import numpy as np
import pandas as pd
import Soil_Carbon_Model as md
import os
import openpyxl
from pandas import *

In [2]:
dt = 1 / 24
nyears = 1000
ndays = nyears * 365
nsim = ndays * 24
nnpp = ndays * 24

#Rainfall from Marked Poisson
rain = md.poisson_process(n = int(nsim), alpha = 16.64/1000, lbd = (0.24 * dt))

tsim = np.linspace(1, nnpp, int(nnpp))

days = np.arange(365)
annual_avg_npp = 996.5

shape = np.sin((np.pi*days)/365)                       

daily_npp = annual_avg_npp * (shape / np.sum(shape))
hourly_npp = np.repeat(daily_npp, 24)

npp = np.tile(hourly_npp,nyears)

In [3]:
print("Maximum Value is:", max(npp))
print("Average value is:", np.mean(npp))

Maximum Value is: 4.288475912207284
Average value is: 2.7301369863013685


In [4]:
total_rainfall = rain.sum()                  # total rainfall in 'm'
print(f"Total rainfall over 450 years = {total_rainfall:.3f} meters")

RPY = (total_rainfall / nyears)*1000         # average rainfall per year in 'mm'
print(f"Average rainfall per year = {RPY:.3f} mm/year")

PET = 1119.8
AI = PET / RPY
print("AI =", AI)

Total rainfall over 450 years = 1453.329 meters
Average rainfall per year = 1453.329 mm/year
AI = 0.7705071366506887


In [5]:
# Compute TI_DI and TI_Texture

KA_final = 9.0132
n_final = 1.7444
R_max = 3.8                   

def compute_TI(AI):
    AI = np.asarray(AI, dtype=float)
    numerator   = AI**n_final
    denominator = KA_final**n_final + AI**n_final
    norm_factor = R_max**n_final / (KA_final**n_final + R_max**n_final)
    TI = (numerator / denominator) / norm_factor

    return np.minimum(TI, 1.0)

# ------------------------------------
# TI FROM SAND (SIGMOID)
# ------------------------------------
sand = 54.0   # ZAN sand (%)
s50 = 25.0
k   = 0.08

def TI_from_sand(sand):
    return 1.0 / (1.0 + np.exp(-k * (sand - s50)))

TI_sand = TI_from_sand(sand)
TI_AI = compute_TI(AI)
TI = (TI_sand + TI_AI)/2

print("TI_DI =", TI_AI)
print("TI_tex =", TI_sand)
print("TI Total =", TI)

TI_DI = 0.07450026713235339
TI_tex = 0.9105199406664386
TI Total = 0.492510103899396


In [6]:
#Hydraulic parameterization for dynamic porosity -------------------------------
alpha1 = 580  #hydraulic conductivity shape parameter
beta1 = 6     #hydraulic conductivity shape parameter
alpha2 = 580  #porosity shape parameter
beta2 = 6     #porosity shape parameter
kstex = 0.80   #m/d,
kstot = 1     #m/d
phitex = 0.43  
phitot = 0.60  

In [7]:
# Simulation for 2150 years at dynamic porosity
sim = md.run_julia(rain, npp, kstex, kstot, phitex, phitot, alpha1, beta1, alpha2, beta2, TI)
sim_year = sim.pivot_table(values=['s', 'phi', 'ksstr', 'P', 'DOC', 'A', 'M', 'B',
                                   'Fa', 'Fpa','Flb','Fl', 'Flm', 'Fpl', 'Fbm','Fld', 'Fal',
                                   'lk','npp_', 'L','Fi', 'Flpom', 'decomp'],
                           index='Years')

print(sim_year)

100%|██████████████████████████████████████████████████████████████████████| 8760000/8760000 [23:14<00:00, 6283.46it/s]


                 A           B         DOC        Fa       Fal       Fbm  \
Years                                                                      
1       939.779860  104.766702  177.888165  0.227396  0.006903  0.768467   
2       965.490065  116.016752  217.238772  0.245837  0.007616  0.850939   
3      1022.098640  134.813676  204.374984  0.253029  0.008032  0.988824   
4      1108.498401  150.340487  184.891687  0.284508  0.009154  1.102724   
5      1134.970582  155.658468  191.897202  0.282329  0.009129  1.141725   
...            ...         ...         ...       ...       ...       ...   
996    1804.587286  172.337080  182.737948  0.451658  0.014731  1.264064   
997    1804.980074  174.446011  186.650401  0.447548  0.014619  1.279529   
998    1805.211512  175.448931  176.778051  0.454515  0.014859  1.286899   
999    1805.258521  168.200774  188.021556  0.438650  0.014267  1.233727   
1000   1805.773747  172.451555  178.410505  0.454876  0.014837  1.264911   

           

In [8]:
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Zanatta\ZAN.csv", index= True, header = True)
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Results\ZAN.csv", index= True, header = True)